# Multimodal Biosignal-Based Immersion Classification

Clean training notebook for EEG, GSR, and PPG binary classification.

## 1. Setup

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from torch.utils.data import DataLoader

from Dataset import BiosignalDataset
from model import FusionModel


## 2. Configuration

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

batch_size = 8
test_batch_size = 32
epochs = 50
lr = 1e-3
patience = 20

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_dir = './data/DATA_SLICED'
split_dir = './data/data_splits'
output_dir = Path('./outputs')
output_dir.mkdir(parents=True, exist_ok=True)

print('Device:', device)


## 3. Dataset & DataLoader

In [ ]:
train_dataset = BiosignalDataset(data_dir, split='train', split_path=split_dir)
val_dataset = BiosignalDataset(data_dir, split='val', split_path=split_dir)
test_dataset = BiosignalDataset(data_dir, split='test', split_path=split_dir)

pin_memory = device.type == 'cuda'
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory)

print('Train:', len(train_dataset), 'Val:', len(val_dataset), 'Test:', len(test_dataset))


## 4. Model

In [ ]:
model = FusionModel().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)


## 5. Training

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for eeg, gsr, ppg, label in loader:
        eeg = eeg.to(device)
        gsr = gsr.to(device)
        ppg = ppg.to(device)
        label = label.to(device).view(-1, 1)

        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(eeg, gsr, ppg)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()

        batch_size_now = label.size(0)
        total_loss += loss.item() * batch_size_now
        total_correct += ((torch.sigmoid(logits) >= 0.5).float() == label).sum().item()
        total_samples += batch_size_now

    return total_loss / total_samples, total_correct / total_samples

@torch.no_grad()
def evaluate_one_epoch(model, loader):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for eeg, gsr, ppg, label in loader:
        eeg = eeg.to(device)
        gsr = gsr.to(device)
        ppg = ppg.to(device)
        label = label.to(device).view(-1, 1)

        logits, _ = model(eeg, gsr, ppg)
        loss = criterion(logits, label)

        batch_size_now = label.size(0)
        total_loss += loss.item() * batch_size_now
        total_correct += ((torch.sigmoid(logits) >= 0.5).float() == label).sum().item()
        total_samples += batch_size_now

    return total_loss / total_samples, total_correct / total_samples


In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
early_stop_counter = 0
checkpoint_path = output_dir / 'best_model.pt'

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate_one_epoch(model, val_loader)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'[Epoch {epoch:03d}] Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0
        torch.save(model.state_dict(), checkpoint_path)
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print('Early stopping triggered.')
            break


## 6. Learning Curves

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'], label='Validation')
plt.title('Loss')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train')
plt.plot(history['val_acc'], label='Validation')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.savefig(output_dir / 'learning_curves.png', dpi=200)
plt.show()


## 7. Test Evaluation

In [ ]:
def load_checkpoint(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)

model.load_state_dict(load_checkpoint(checkpoint_path, device))
model.eval()

y_true, y_prob = [], []
with torch.no_grad():
    for eeg, gsr, ppg, label in test_loader:
        eeg = eeg.to(device)
        gsr = gsr.to(device)
        ppg = ppg.to(device)
        logits, _ = model(eeg, gsr, ppg)
        prob = torch.sigmoid(logits).view(-1)
        y_prob.extend(prob.cpu().numpy())
        y_true.extend(label.numpy())

y_true = np.asarray(y_true)
y_prob = np.asarray(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

print('Accuracy:', accuracy_score(y_true, y_pred))
print('\nConfusion Matrix\n', confusion_matrix(y_true, y_pred))
print('\nClassification Report\n', classification_report(y_true, y_pred, digits=4))
